# DICOM Linkage Analyzer

This notebook analyzes DICOM files in a patient root directory and builds UID-based linkage trees between RT modalities (RTSTRUCT, RTPLAN, RTDOSE) and MRI series.

## Features

- **Pattern-based folder detection**: Identifies modality folders by searching for patterns in folder names (e.g., `_MR_`, `_RTst_`, `_RTPLAN_`, `_RTDOSE_`)
- **DICOM header verification**: Verifies folder modality by reading DICOM headers (not just folder names)
- **Multiple folder support**: Handles multiple folders per modality (e.g., multiple MR series, multiple RTSTRUCT files)
- **File counting**: Reports total files and readable DICOM files per modality
- **Minimal metadata parsing**: Reads only DICOM headers (no pixel data) for efficiency
- **UID-based linkage**: Builds links between RT modalities using DICOM UID references
- **Plan-centric tree**: Outputs linkage tree organized by RTPLAN with folder counts
- **Orphan detection**: Identifies unlinked objects and missing references

In [1]:
import pydicom
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Any
import json
from dataclasses import dataclass, asdict
from datetime import datetime

## Configuration

Configure the patient root directory and modality folder name patterns.

**How it works:**
1. The notebook searches for folders containing the specified patterns in their names
2. For each matching folder, it reads DICOM headers to verify the modality matches
3. Only folders with verified DICOM files of the expected modality are included
4. Multiple folders per modality are supported and aggregated

In [2]:
# Configuration: Patient root directory
# PATIENT_ROOT = Path("/path/to/patient/root")  # UPDATE THIS PATH
PATIENT_ROOT = Path("/database/brainmets/dicom/SBRT_research_mim_export_20251209_organized/SRS1733/2002-06__Studies")

# Modality folder name patterns (substrings to search for in folder names)
# Folders will be identified by these patterns, then verified by reading DICOM headers
MODALITY_PATTERNS = {
    'MRI': ['_MR_'],  # Matches folders like "SRS1733_SRS1733_MR_..."
    'RTSTRUCT': ['_RTst_'],  # Matches folders like "SRS1733_SRS1733_RTst_..."
    'RTPLAN': ['_RTPLAN_'],  # Matches folders like "SRS1733_SRS1733_RTPLAN_..."
    'RTDOSE': ['_RTDOSE_']  # Matches folders like "SRS1733_SRS1733_RTDOSE_..."
}

# Expected DICOM modality tags (for verification)
MODALITY_TAGS = {
    'MRI': 'MR',
    'RTSTRUCT': 'RTSTRUCT',
    'RTPLAN': 'RTPLAN',
    'RTDOSE': 'RTDOSE'
}

## Core Functions

### Data Structures

In [3]:
@dataclass
class FolderCountInfo:
    """Information about files in a modality folder."""
    folder_path: str
    n_files_total: int
    n_dicom_readable: int
    representative_file: Optional[str] = None
    files: List[str] = None
    
    def __post_init__(self):
        if self.files is None:
            self.files = []


@dataclass
class MRIInfo:
    """Representative MRI series information."""
    patient_id: str
    study_instance_uid: str
    series_instance_uid: str
    frame_of_reference_uid: Optional[str]
    sop_instance_uid: str
    representative_file: str
    modality: str = "MR"


@dataclass
class RTStructInfo:
    """RTSTRUCT DICOM information."""
    sop_instance_uid: str
    patient_id: str
    study_instance_uid: str
    series_instance_uid: str
    frame_of_reference_uid: Optional[str]
    file_path: str
    referenced_series_uids: List[str]
    referenced_image_sops: List[str]
    referenced_frame_of_reference_uids: List[str]


@dataclass
class RTPlanInfo:
    """RTPLAN DICOM information."""
    sop_instance_uid: str
    patient_id: str
    study_instance_uid: str
    series_instance_uid: str
    file_path: str
    referenced_rtstruct_sop: Optional[str]


@dataclass
class RTDoseInfo:
    """RTDOSE DICOM information."""
    sop_instance_uid: str
    patient_id: str
    study_instance_uid: str
    series_instance_uid: str
    file_path: str
    referenced_rtplan_sop: Optional[str]
    dose_summation_type: Optional[str]

### Folder Scanning Functions

In [4]:
def find_modality_folders(patient_root: Path, modality_key: str, patterns: List[str]) -> List[Path]:
    """
    Find all folders matching modality patterns by searching folder name substrings.
    
    Args:
        patient_root: Root directory to search
        modality_key: Key for modality ('MRI', 'RTSTRUCT', etc.)
        patterns: List of substring patterns to search for in folder names
        
    Returns:
        List of Path objects to matching folders
    """
    matching_folders = []
    
    if not patient_root.exists():
        return matching_folders
    
    for folder_path in patient_root.iterdir():
        if not folder_path.is_dir():
            continue
        
        folder_name = folder_path.name
        # Check if folder name contains any of the patterns
        for pattern in patterns:
            if pattern in folder_name:
                matching_folders.append(folder_path)
                break  # Found a match, no need to check other patterns
    
    return matching_folders


def verify_folder_modality(folder_path: Path, expected_modality: str) -> Tuple[bool, List[str]]:
    """
    Verify that a folder contains DICOM files of the expected modality.
    Collects all valid files that match the expected modality.
    
    Args:
        folder_path: Path to folder to verify
        expected_modality: Expected DICOM modality tag (e.g., 'MR', 'RTSTRUCT')
        
    Returns:
        Tuple of (is_valid, list_of_valid_file_paths)
        is_valid is True if at least one file matches the expected modality
    """
    valid_files = []
    
    if not folder_path.exists() or not folder_path.is_dir():
        return False, []
    
    # List all files
    all_files = [f for f in folder_path.iterdir() if f.is_file()]
    
    if not all_files:
        return False, []
    
    # Check all files and collect those matching the expected modality
    for file_path in sorted(all_files):
        try:
            ds = pydicom.dcmread(file_path, stop_before_pixels=True, force=True)
            
            # Check if modality matches
            if not hasattr(ds, 'Modality'):
                continue
            
            if ds.Modality == expected_modality:
                valid_files.append(str(file_path))
        except Exception as e:
            # Skip files that can't be read or don't have the expected modality
            continue
    
    # Folder is valid if we found at least one matching file
    is_valid = len(valid_files) > 0
    return is_valid, valid_files


def scan_patient(patient_root: Path, modality_patterns: Dict[str, List[str]], 
                 modality_tags: Dict[str, str]) -> Dict[str, FolderCountInfo]:
    """
    Scan patient root directory for modality folders using name patterns and verify by DICOM headers.
    
    Args:
        patient_root: Root directory containing modality subfolders
        modality_patterns: Dictionary mapping modality keys to folder name patterns
        modality_tags: Dictionary mapping modality keys to expected DICOM modality tags
        
    Returns:
        Dictionary mapping modality keys to FolderCountInfo objects
    """
    folder_counts = {}
    
    for modality_key, patterns in modality_patterns.items():
        # Find all folders matching the patterns
        matching_folders = find_modality_folders(patient_root, modality_key, patterns)
        expected_modality = modality_tags.get(modality_key, '')
        
        if not matching_folders:
            folder_counts[modality_key] = FolderCountInfo(
                folder_path="",
                n_files_total=0,
                n_dicom_readable=0
            )
            continue
        
        # Collect all valid files from all matching folders
        all_valid_files = []
        total_files = 0
        verified_folders = []
        
        for folder_path in matching_folders:
            is_valid, valid_files = verify_folder_modality(folder_path, expected_modality)
            
            if is_valid:
                verified_folders.append(str(folder_path))
                all_valid_files.extend(valid_files)
                
                # Count all files in folder (not just valid DICOM)
                folder_files = [f for f in folder_path.iterdir() if f.is_file()]
                total_files += len(folder_files)
        
        # Sort all files
        all_valid_files_sorted = sorted(set(all_valid_files))  # Remove duplicates if any
        
        # Use first verified folder as representative path (or concatenate if multiple)
        representative_path = verified_folders[0] if verified_folders else ""
        if len(verified_folders) > 1:
            representative_path += f" (+ {len(verified_folders)-1} more)"
        
        folder_counts[modality_key] = FolderCountInfo(
            folder_path=representative_path,
            n_files_total=total_files,
            n_dicom_readable=len(all_valid_files_sorted),
            files=all_valid_files_sorted
        )
    
    return folder_counts

### DICOM Reading Functions

In [5]:
def read_representative_mri(mri_files: List[str]) -> Optional[MRIInfo]:
    """
    Read the first valid DICOM file from MRI folder as representative.
    
    Args:
        mri_files: List of file paths in MRI folder (sorted)
        
    Returns:
        MRIInfo object or None if no readable DICOM found
    """
    for file_path_str in mri_files:
        try:
            ds = pydicom.dcmread(file_path_str, stop_before_pixels=True, force=True)
            
            # Verify it's an MR image
            if not hasattr(ds, 'Modality') or ds.Modality != 'MR':
                continue
            
            # Extract required metadata
            patient_id = str(ds.get('PatientID', 'UNKNOWN'))
            study_uid = str(ds.get('StudyInstanceUID', ''))
            series_uid = str(ds.get('SeriesInstanceUID', ''))
            sop_uid = str(ds.get('SOPInstanceUID', ''))
            frame_uid = ds.get('FrameOfReferenceUID', None)
            if frame_uid is not None:
                frame_uid = str(frame_uid)
            
            return MRIInfo(
                patient_id=patient_id,
                study_instance_uid=study_uid,
                series_instance_uid=series_uid,
                frame_of_reference_uid=frame_uid,
                sop_instance_uid=sop_uid,
                representative_file=file_path_str
            )
        except Exception as e:
            continue
    
    return None


def read_rt_objects(files: List[str], modality: str) -> List[Any]:
    """
    Read all RT DICOM objects (RTSTRUCT, RTPLAN, or RTDOSE).
    
    Args:
        files: List of file paths
        modality: Expected modality ('RTSTRUCT', 'RTPLAN', or 'RTDOSE')
        
    Returns:
        List of RTStructInfo, RTPlanInfo, or RTDoseInfo objects
    """
    rt_objects = []
    
    for file_path_str in files:
        try:
            ds = pydicom.dcmread(file_path_str, stop_before_pixels=True, force=True)
            
            # Verify modality matches
            if not hasattr(ds, 'Modality') or ds.Modality != modality:
                continue
            
            # Common metadata
            patient_id = str(ds.get('PatientID', 'UNKNOWN'))
            study_uid = str(ds.get('StudyInstanceUID', ''))
            series_uid = str(ds.get('SeriesInstanceUID', ''))
            sop_uid = str(ds.get('SOPInstanceUID', ''))
            
            if modality == 'RTSTRUCT':
                # Extract RTSTRUCT-specific references
                frame_uid = ds.get('FrameOfReferenceUID', None)
                if frame_uid is not None:
                    frame_uid = str(frame_uid)
                
                # ReferencedFrameOfReferenceSequence (3006,0010)
                ref_series_uids = []
                ref_image_sops = []
                ref_frame_uids = []
                
                ref_frame_seq = ds.get('ReferencedFrameOfReferenceSequence', [])
                if ref_frame_seq:
                    # Handle pydicom Sequence object
                    if hasattr(ref_frame_seq, '__iter__'):
                        ref_frame_seq = list(ref_frame_seq)
                    else:
                        ref_frame_seq = [ref_frame_seq]
                else:
                    ref_frame_seq = []
                
                for ref_frame_item in ref_frame_seq:
                    ref_frame_uid = ref_frame_item.get('FrameOfReferenceUID', None)
                    if ref_frame_uid:
                        ref_frame_uids.append(str(ref_frame_uid))
                    
                    # RTReferencedStudySequence (3006,0012)
                    ref_study_seq = ref_frame_item.get('RTReferencedStudySequence', [])
                    if ref_study_seq:
                        if hasattr(ref_study_seq, '__iter__') and not isinstance(ref_study_seq, str):
                            ref_study_seq = list(ref_study_seq)
                        else:
                            ref_study_seq = [ref_study_seq]
                    else:
                        ref_study_seq = []
                    
                    for ref_study_item in ref_study_seq:
                        # RTReferencedSeriesSequence (3006,0014)
                        ref_series_seq = ref_study_item.get('RTReferencedSeriesSequence', [])
                        if ref_series_seq:
                            if hasattr(ref_series_seq, '__iter__') and not isinstance(ref_series_seq, str):
                                ref_series_seq = list(ref_series_seq)
                            else:
                                ref_series_seq = [ref_series_seq]
                        else:
                            ref_series_seq = []
                        
                        for ref_series_item in ref_series_seq:
                            series_uid_ref = ref_series_item.get('SeriesInstanceUID', None)
                            if series_uid_ref:
                                ref_series_uids.append(str(series_uid_ref))
                            
                            # ContourImageSequence (3006,0016)
                            contour_seq = ref_series_item.get('ContourImageSequence', [])
                            if contour_seq:
                                if hasattr(contour_seq, '__iter__') and not isinstance(contour_seq, str):
                                    contour_seq = list(contour_seq)
                                else:
                                    contour_seq = [contour_seq]
                            else:
                                contour_seq = []
                            
                            for contour_item in contour_seq:
                                image_sop = contour_item.get('ReferencedSOPInstanceUID', None)
                                if image_sop:
                                    ref_image_sops.append(str(image_sop))
                
                rt_objects.append(RTStructInfo(
                    sop_instance_uid=sop_uid,
                    patient_id=patient_id,
                    study_instance_uid=study_uid,
                    series_instance_uid=series_uid,
                    frame_of_reference_uid=frame_uid,
                    file_path=file_path_str,
                    referenced_series_uids=list(set(ref_series_uids)),  # Deduplicate
                    referenced_image_sops=list(set(ref_image_sops)),  # Deduplicate
                    referenced_frame_of_reference_uids=list(set(ref_frame_uids))  # Deduplicate
                ))
            
            elif modality == 'RTPLAN':
                # ReferencedStructureSetSequence (300C,0060)
                ref_rtstruct_sop = None
                ref_struct_seq = ds.get('ReferencedStructureSetSequence', [])
                if ref_struct_seq:
                    # Handle pydicom Sequence object
                    if hasattr(ref_struct_seq, '__iter__') and not isinstance(ref_struct_seq, str):
                        ref_struct_seq = list(ref_struct_seq)
                    else:
                        ref_struct_seq = [ref_struct_seq]
                    
                    if ref_struct_seq:
                        ref_struct_item = ref_struct_seq[0]  # Usually one
                        ref_rtstruct_sop = ref_struct_item.get('ReferencedSOPInstanceUID', None)
                        if ref_rtstruct_sop:
                            ref_rtstruct_sop = str(ref_rtstruct_sop)
                
                rt_objects.append(RTPlanInfo(
                    sop_instance_uid=sop_uid,
                    patient_id=patient_id,
                    study_instance_uid=study_uid,
                    series_instance_uid=series_uid,
                    file_path=file_path_str,
                    referenced_rtstruct_sop=ref_rtstruct_sop
                ))
            
            elif modality == 'RTDOSE':
                # ReferencedRTPlanSequence (300C,0002)
                ref_rtplan_sop = None
                ref_plan_seq = ds.get('ReferencedRTPlanSequence', [])
                if ref_plan_seq:
                    # Handle pydicom Sequence object
                    if hasattr(ref_plan_seq, '__iter__') and not isinstance(ref_plan_seq, str):
                        ref_plan_seq = list(ref_plan_seq)
                    else:
                        ref_plan_seq = [ref_plan_seq]
                    
                    if ref_plan_seq:
                        ref_plan_item = ref_plan_seq[0]  # Usually one
                        ref_rtplan_sop = ref_plan_item.get('ReferencedSOPInstanceUID', None)
                        if ref_rtplan_sop:
                            ref_rtplan_sop = str(ref_rtplan_sop)
                
                # DoseSummationType (3004,000A)
                dose_summation_type = ds.get('DoseSummationType', None)
                if dose_summation_type:
                    dose_summation_type = str(dose_summation_type)
                
                rt_objects.append(RTDoseInfo(
                    sop_instance_uid=sop_uid,
                    patient_id=patient_id,
                    study_instance_uid=study_uid,
                    series_instance_uid=series_uid,
                    file_path=file_path_str,
                    referenced_rtplan_sop=ref_rtplan_sop,
                    dose_summation_type=dose_summation_type
                ))
        
        except Exception as e:
            print(f"Warning: Could not read {file_path_str}: {e}")
            continue
    
    return rt_objects

### Linkage Analysis Functions

In [6]:
def build_link_graph(
    mri_info: Optional[MRIInfo],
    rtstructs: List[RTStructInfo],
    rtplans: List[RTPlanInfo],
    rtdoses: List[RTDoseInfo]
) -> Tuple[Dict[str, Any], Dict[str, List[str]]]:
    """
    Build linkage graph and identify orphans/missing references.
    
    Returns:
        Tuple of (summary_dict, orphans_dict)
    """
    # Build indexes
    rtstruct_by_sop = {rt.sop_instance_uid: rt for rt in rtstructs}
    rtplan_by_sop = {rt.sop_instance_uid: rt for rt in rtplans}
    rtdose_by_sop = {rt.sop_instance_uid: rt for rt in rtdoses}
    
    # Track which RTSTRUCTs are referenced by RTPLANs
    referenced_rtstruct_sops = set()
    for rtplan in rtplans:
        if rtplan.referenced_rtstruct_sop:
            referenced_rtstruct_sops.add(rtplan.referenced_rtstruct_sop)
    
    # Track which RTPLANs are referenced by RTDOSEs
    referenced_rtplan_sops = set()
    for rtdose in rtdoses:
        if rtdose.referenced_rtplan_sop:
            referenced_rtplan_sops.add(rtdose.referenced_rtplan_sop)
    
    # Identify orphans
    orphans = {
        'orphan_rtstruct': [],
        'orphan_rtplan': [],
        'orphan_rtdose': [],
        'rtstruct_missing_images': [],
        'rtdose_missing_rtplan': [],
        'rtplan_missing_rtstruct': []
    }
    
    # Orphan RTSTRUCTs (not referenced by any RTPLAN)
    for rtstruct in rtstructs:
        if rtstruct.sop_instance_uid not in referenced_rtstruct_sops:
            orphans['orphan_rtstruct'].append(rtstruct.sop_instance_uid)
    
    # Orphan RTPLANs (not referenced by any RTDOSE)
    for rtplan in rtplans:
        if rtplan.sop_instance_uid not in referenced_rtplan_sops:
            orphans['orphan_rtplan'].append(rtplan.sop_instance_uid)
    
    # RTDOSE referencing missing RTPLAN
    for rtdose in rtdoses:
        if rtdose.referenced_rtplan_sop:
            if rtdose.referenced_rtplan_sop not in rtplan_by_sop:
                orphans['orphan_rtdose'].append(rtdose.sop_instance_uid)
                orphans['rtdose_missing_rtplan'].append({
                    'rtdose_sop': rtdose.sop_instance_uid,
                    'missing_rtplan_sop': rtdose.referenced_rtplan_sop
                })
    
    # RTPLAN referencing missing RTSTRUCT
    for rtplan in rtplans:
        if rtplan.referenced_rtstruct_sop:
            if rtplan.referenced_rtstruct_sop not in rtstruct_by_sop:
                orphans['rtplan_missing_rtstruct'].append({
                    'rtplan_sop': rtplan.sop_instance_uid,
                    'missing_rtstruct_sop': rtplan.referenced_rtstruct_sop
                })
    
    # RTSTRUCT → MRI linkage validation
    if mri_info:
        for rtstruct in rtstructs:
            # Check if RTSTRUCT references the MRI series
            mri_series_matched = False
            if mri_info.series_instance_uid in rtstruct.referenced_series_uids:
                mri_series_matched = True
            
            # Check FrameOfReferenceUID match
            frame_uid_matched = False
            if mri_info.frame_of_reference_uid and rtstruct.frame_of_reference_uid:
                if mri_info.frame_of_reference_uid == rtstruct.frame_of_reference_uid:
                    frame_uid_matched = True
            
            # If RTSTRUCT references different series, flag it
            if rtstruct.referenced_series_uids and not mri_series_matched:
                orphans['rtstruct_missing_images'].append({
                    'rtstruct_sop': rtstruct.sop_instance_uid,
                    'referenced_series': rtstruct.referenced_series_uids,
                    'mri_series': mri_info.series_instance_uid,
                    'frame_uid_match': frame_uid_matched
                })
    
    # Build summary structure
    summary = {
        'patient_id': mri_info.patient_id if mri_info else 'UNKNOWN',
        'mri': mri_info.__dict__ if mri_info else None,
        'rtstructs': [rt.__dict__ for rt in rtstructs],
        'rtplans': [rt.__dict__ for rt in rtplans],
        'rtdoses': [rt.__dict__ for rt in rtdoses]
    }
    
    return summary, orphans

### Report Generation Functions

In [7]:
def print_folder_count_summary(folder_counts: Dict[str, FolderCountInfo]):
    """Print folder count summary report."""
    print("=" * 80)
    print("FOLDER COUNT SUMMARY")
    print("=" * 80)
    print()
    
    for modality, info in folder_counts.items():
        if info.folder_path:
            print(f"{modality}:")
            print(f"  Folder: {info.folder_path}")
            print(f"  Total files: {info.n_files_total}")
            print(f"  Readable DICOM: {info.n_dicom_readable}")
            if info.representative_file:
                print(f"  Representative file: {info.representative_file}")
            print()
        else:
            print(f"{modality}: FOLDER NOT FOUND")
            print()
    
    print("=" * 80)
    print()


def print_linkage_tree(
    folder_counts: Dict[str, FolderCountInfo],
    mri_info: Optional[MRIInfo],
    rtstructs: List[RTStructInfo],
    rtplans: List[RTPlanInfo],
    rtdoses: List[RTDoseInfo]
):
    """Print plan-centric linkage tree."""
    print("=" * 80)
    print("LINKAGE TREE")
    print("=" * 80)
    print()
    
    patient_id = mri_info.patient_id if mri_info else 'UNKNOWN'
    print(f"Patient: {patient_id}")
    print()
    
    # MRI section
    mri_count = folder_counts.get('MRI', FolderCountInfo("", 0, 0))
    if mri_info:
        print(f"MRI (folder count {mri_count.n_files_total}):")
        print(f"  SeriesUID: {mri_info.series_instance_uid}")
        print(f"  StudyUID: {mri_info.study_instance_uid}")
        if mri_info.frame_of_reference_uid:
            print(f"  FrameOfReferenceUID: {mri_info.frame_of_reference_uid}")
        print(f"  Representative file: {mri_info.representative_file}")
        print()
    elif mri_count.folder_path:
        print(f"MRI (folder count {mri_count.n_files_total}): NO READABLE DICOM FOUND")
        print()
    
    # RTSTRUCT section
    rtstruct_count = folder_counts.get('RTSTRUCT', FolderCountInfo("", 0, 0))
    for rtstruct in rtstructs:
        print(f"RTSTRUCT (count {rtstruct_count.n_files_total}):")
        print(f"  SOP: {rtstruct.sop_instance_uid}")
        print(f"  File: {Path(rtstruct.file_path).name}")
        if rtstruct.referenced_series_uids:
            print(f"  → References MRI SeriesUIDs: {rtstruct.referenced_series_uids}")
        if rtstruct.referenced_image_sops:
            print(f"  → References Image SOPs: {len(rtstruct.referenced_image_sops)} images")
        print()
    
    # RTPLAN section (plan-centric)
    rtplan_count = folder_counts.get('RTPLAN', FolderCountInfo("", 0, 0))
    for rtplan in rtplans:
        print(f"RTPLAN (count {rtplan_count.n_files_total}):")
        print(f"  SOP: {rtplan.sop_instance_uid}")
        print(f"  File: {Path(rtplan.file_path).name}")
        if rtplan.referenced_rtstruct_sop:
            print(f"  → References RTSTRUCT SOP: {rtplan.referenced_rtstruct_sop}")
        else:
            print(f"  → References RTSTRUCT SOP: NONE")
        
        # Find linked RTDOSEs
        linked_rtdoses = [
            rtd for rtd in rtdoses 
            if rtd.referenced_rtplan_sop == rtplan.sop_instance_uid
        ]
        if linked_rtdoses:
            print(f"  ← Referenced by RTDOSE(s):")
            for rtd in linked_rtdoses:
                print(f"     - {rtd.sop_instance_uid} (DoseSummationType: {rtd.dose_summation_type or 'N/A'})")
        print()
    
    # RTDOSE section
    rtdose_count = folder_counts.get('RTDOSE', FolderCountInfo("", 0, 0))
    for rtdose in rtdoses:
        print(f"RTDOSE (count {rtdose_count.n_files_total}):")
        print(f"  SOP: {rtdose.sop_instance_uid}")
        print(f"  File: {Path(rtdose.file_path).name}")
        if rtdose.referenced_rtplan_sop:
            print(f"  → References RTPLAN SOP: {rtdose.referenced_rtplan_sop}")
        print(f"  DoseSummationType: {rtdose.dose_summation_type or 'N/A'}")
        print()
    
    print("=" * 80)
    print()


def print_orphan_report(orphans: Dict[str, List], folder_counts: Dict[str, FolderCountInfo]):
    """Print orphan and missing link report."""
    print("=" * 80)
    print("ORPHAN / MISSING LINK REPORT")
    print("=" * 80)
    print()
    
    # Orphan RTSTRUCTs
    if orphans['orphan_rtstruct']:
        print(f"Orphan RTSTRUCT (not referenced by any RTPLAN): {len(orphans['orphan_rtstruct'])}")
        for sop in orphans['orphan_rtstruct']:
            print(f"  - {sop}")
        print()
    else:
        print("Orphan RTSTRUCT: None")
        print()
    
    # Orphan RTPLANs
    if orphans['orphan_rtplan']:
        print(f"Orphan RTPLAN (not referenced by any RTDOSE): {len(orphans['orphan_rtplan'])}")
        for sop in orphans['orphan_rtplan']:
            print(f"  - {sop}")
        print()
    else:
        print("Orphan RTPLAN: None")
        print()
    
    # RTDOSE referencing missing RTPLAN
    if orphans['rtdose_missing_rtplan']:
        print(f"RTDOSE referencing missing RTPLAN: {len(orphans['rtdose_missing_rtplan'])}")
        for item in orphans['rtdose_missing_rtplan']:
            print(f"  - RTDOSE {item['rtdose_sop']} → missing RTPLAN {item['missing_rtplan_sop']}")
        print()
    else:
        print("RTDOSE missing RTPLAN references: None")
        print()
    
    # RTPLAN referencing missing RTSTRUCT
    if orphans['rtplan_missing_rtstruct']:
        print(f"RTPLAN referencing missing RTSTRUCT: {len(orphans['rtplan_missing_rtstruct'])}")
        for item in orphans['rtplan_missing_rtstruct']:
            print(f"  - RTPLAN {item['rtplan_sop']} → missing RTSTRUCT {item['missing_rtstruct_sop']}")
        print()
    else:
        print("RTPLAN missing RTSTRUCT references: None")
        print()
    
    # RTSTRUCT missing images
    if orphans['rtstruct_missing_images']:
        print(f"RTSTRUCT referencing different MRI series: {len(orphans['rtstruct_missing_images'])}")
        for item in orphans['rtstruct_missing_images']:
            print(f"  - RTSTRUCT {item['rtstruct_sop']}")
            print(f"    Referenced series: {item['referenced_series']}")
            print(f"    MRI representative series: {item['mri_series']}")
            print(f"    FrameOfReferenceUID match: {item['frame_uid_match']}")
        print()
    else:
        print("RTSTRUCT missing images: None")
        print()
    
    # Missing/empty folders
    missing_folders = []
    empty_folders = []
    for modality, info in folder_counts.items():
        if not info.folder_path:
            missing_folders.append(modality)
        elif info.n_files_total == 0:
            empty_folders.append(modality)
    
    if missing_folders:
        print(f"Missing folders: {', '.join(missing_folders)}")
    else:
        print("Missing folders: None")
    
    if empty_folders:
        print(f"Empty folders: {', '.join(empty_folders)}")
    else:
        print("Empty folders: None")
    
    print()
    print("=" * 80)
    print()


def export_json(summary: Dict, orphans: Dict, folder_counts: Dict, output_path: Path):
    """Export analysis results to JSON file."""
    export_data = {
        'timestamp': datetime.now().isoformat(),
        'folder_counts': {k: asdict(v) for k, v in folder_counts.items()},
        'linkage_summary': summary,
        'orphans': orphans
    }
    
    with open(output_path, 'w') as f:
        json.dump(export_data, f, indent=2)
    
    print(f"Results exported to: {output_path}")

## Main Analysis Workflow

Run the complete analysis pipeline.

In [8]:
def analyze_patient(patient_root: Path, modality_patterns: Dict[str, List[str]], 
                    modality_tags: Dict[str, str]) -> Tuple[Dict, Dict, Dict]:
    """
    Complete analysis pipeline for a patient directory.
    
    Returns:
        Tuple of (folder_counts, summary, orphans, mri_info, rtstructs, rtplans, rtdoses)
    """
    # Step 1: Scan folders and count files
    print("Step 1: Scanning folders and counting files...")
    folder_counts = scan_patient(patient_root, modality_patterns, modality_tags)
    
    # Step 2: Read DICOM headers
    print("\nStep 2: Reading DICOM headers...")
    
    # MRI: Read representative file
    mri_info = None
    mri_count = folder_counts.get('MRI')
    if mri_count and mri_count.files:
        mri_info = read_representative_mri(mri_count.files)
        if mri_info:
            folder_counts['MRI'].representative_file = mri_info.representative_file
    
    # RTSTRUCT: Read all files
    rtstructs = []
    rtstruct_count = folder_counts.get('RTSTRUCT')
    if rtstruct_count and rtstruct_count.files:
        rtstructs = read_rt_objects(rtstruct_count.files, 'RTSTRUCT')
    
    # RTPLAN: Read all files
    rtplans = []
    rtplan_count = folder_counts.get('RTPLAN')
    if rtplan_count and rtplan_count.files:
        rtplans = read_rt_objects(rtplan_count.files, 'RTPLAN')
    
    # RTDOSE: Read all files
    rtdoses = []
    rtdose_count = folder_counts.get('RTDOSE')
    if rtdose_count and rtdose_count.files:
        rtdoses = read_rt_objects(rtdose_count.files, 'RTDOSE')
    
    # Step 3: Build linkage graph
    print("\nStep 3: Building linkage graph...")
    summary, orphans = build_link_graph(mri_info, rtstructs, rtplans, rtdoses)
    
    return folder_counts, summary, orphans, mri_info, rtstructs, rtplans, rtdoses

## Execute Analysis

Update `PATIENT_ROOT` in the configuration cell above, then run this cell.

In [9]:
# Execute analysis
if PATIENT_ROOT.exists():
    folder_counts, summary, orphans, mri_info, rtstructs, rtplans, rtdoses = analyze_patient(
        PATIENT_ROOT, 
        MODALITY_PATTERNS,
        MODALITY_TAGS
    )
    
    # Print reports
    print_folder_count_summary(folder_counts)
    print_linkage_tree(folder_counts, mri_info, rtstructs, rtplans, rtdoses)
    print_orphan_report(orphans, folder_counts)
    
    # Optional: Export to JSON
    # output_json = PATIENT_ROOT / "linkage_analysis.json"
    # export_json(summary, orphans, folder_counts, output_json)
else:
    print(f"ERROR: Patient root directory does not exist: {PATIENT_ROOT}")
    print("Please update PATIENT_ROOT in the configuration cell above.")

Step 1: Scanning folders and counting files...

Step 2: Reading DICOM headers...

Step 3: Building linkage graph...
FOLDER COUNT SUMMARY

MRI:
  Folder: /database/brainmets/dicom/SBRT_research_mim_export_20251209_organized/SRS1733/2002-06__Studies/SRS1733_SRS1733_MR_2002-06-26_000000_._axial_n52__00000 (+ 1 more)
  Total files: 104
  Readable DICOM: 104
  Representative file: /database/brainmets/dicom/SBRT_research_mim_export_20251209_organized/SRS1733/2002-06__Studies/SRS1733_SRS1733_MR_2002-06-26_000000_._axial_n52__00000/2.16.840.1.114362.1.12046989.25631758973.631607717.1022.4238.dcm

RTSTRUCT:
  Folder: /database/brainmets/dicom/SBRT_research_mim_export_20251209_organized/SRS1733/2002-06__Studies/SRS1733_SRS1733_RTst_2002-06-26_000000_._Brain-MS-noreRT_n1__00000 (+ 5 more)
  Total files: 6
  Readable DICOM: 6

RTPLAN:
  Folder: /database/brainmets/dicom/SBRT_research_mim_export_20251209_organized/SRS1733/2002-06__Studies/SRS1733_SRS1733_RTPLAN_2002-06-26_000000_._._n1__00001 (+ 1 